In [ ]:
df_xpr = pd.read_csv("../data/TGCA_BRCA_expression_matrix.TPM.csv.gz" , index_col = 0)
df_xpr.head()

In [ ]:
df_clinical = pd.read_csv("../data/TGCA_BRCA_clinical_filtered.small.csv",index_col=0)
df_clinical.head()

In [ ]:
y = df_clinical.poor_prognosis
X_xpr = df_xpr.loc[ :, df_clinical.index].transpose() 
X_xpr.shape

In [ ]:
y.value_counts()

In [ ]:
## how long to train a single model with so many features?
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

Xs = StandardScaler().fit_transform(X_xpr)

lr = LogisticRegression( max_iter = 1000)
%time lr.fit(  Xs , y )


In [ ]:
## how many of these have a coefficient ?
( lr.coef_ != 0  ).sum()

In [ ]:
from sklearn.metrics import accuracy_score

## very likely overfit model
accuracy_score(y , lr.predict( Xs ))

In [ ]:
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html#sklearn.feature_selection.mutual_info_classif
#from sklearn.feature_selection import mutual_info_classif

## takes very long
#MI = mutual_info_classif(X_xpr , y)

In [ ]:
from sklearn.feature_selection import chi2

%time CHI2 = pd.Series( chi2(X_xpr , y)[1] , index = X_xpr.columns )# get the pvalues

In [ ]:
import numpy as np
sns.histplot(np.log10(CHI2))

In [ ]:
X_xpr.var()[ np.log10(CHI2) < -50 ]

In [ ]:
import matplotlib.pyplot as plt
fig,ax = plt.subplots(2,1, sharex=True)
for i, gene in enumerate(['ENSG00000002726.21' , 'ENSG00000002586.20']):
    
    sns.stripplot( x = X_xpr[gene] , y=y , orient='horizontal' , ax=ax[i],  alpha = 0.8)
    sns.boxplot( x = X_xpr[gene] , y=y , orient='horizontal' , ax=ax[i] , fliersize= 0 , color ='grey')
    ax[i].set_xlabel("TPM")
    ax[i].set_title(f'{gene} - pvalue = {CHI2[gene]:.1e}')

fig.tight_layout()

In [ ]:
sns.histplot(np.log10(1+X_xpr.var()) )
plt.xlabel("log10( 1 + expression variance)")
plt.title(f"fraction of genes with 0 variance: {(X_xpr.var()==0).mean():.3f}")

In [ ]:
threshold = X_xpr.var().quantile(0.9)
top10_percent = X_xpr.var() > threshold

In [ ]:

sns.scatterplot( x = np.log10(1+X_xpr.mean()) , y= np.log10(1+X_xpr.var()) , hue = top10_percent )
plt.xlabel('log( 1 + mean expression )')
plt.ylabel('log( 1 + mean variance )')

In [ ]:
from sklearn.feature_selection import VarianceThreshold

threshold = X_xpr.var(ddof = 0).quantile(0.9)

VT = VarianceThreshold(threshold)
X_filtered = VT.fit_transform(X_xpr)
X_filtered.shape

In [ ]:
## alternative with SelectPercentile
## a bit more adavcned because you need a lambda function
## but as you do not have to compute the threshold beforehand you can fit this more easily into pipeline and 
## proper cross-validation loops

from sklearn.feature_selection import SelectPercentile


VT = SelectPercentile( score_func = lambda x,_ : np.var(x , axis = 0) ,
                       percentile = 10
                     )
X_filtered = VT.fit_transform(X_xpr)
X_filtered.shape

In [ ]:
X_filtered = pd.DataFrame( X_filtered , 
                          columns= VT.get_feature_names_out() , 
                          index = X_xpr.index)
X_filtered.head()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

lr_ppl = Pipeline( [('scale',StandardScaler()),
                    ('model',LogisticRegression())] )

%time cross_val_score(lr_ppl , X_filtered , y , cv = 5 , scoring = 'accuracy')

## exercise

---

1. use different thresholds for variance selection, and evaluate them with `cross_val_score`. 

      * the top 50% variables genes
      * the top 0.1% variable genes
     
    Which one yields the better cross-validated accuracy?
     
--- 

2. select the top 10% variable genes, then use [SelectKbest](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html#sklearn.feature_selection.SelectKBest) (use `chi2` as scoring function) to get the top 500 genes. 
    Which cross-validated accuracy fo you get?


--- **correction** ---


1. use different thresholds for variance selection, and evaluate them with `cross_val_score`. 

      * the top 50% variables genes
      * the top 0.1% variable genes
     
    Which one yields the better cross-validated accuracy?

In [ ]:
# %load -r -23 solutions/solution_univariate.py

2. select the top 10% variable genes, then use [SelectKbest](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html#sklearn.feature_selection.SelectKBest) (use `chi2` as scoring function) to get the top 500 genes. 
    Which cross-validated accuracy fo you get?


In [ ]:
# %load -r 24-43 solutions/solution_univariate.py

2. cleaner version:

In [ ]:
# %load -r 44- solutions/solution_univariate.py